# Rust 123: Unit Tests — Practice

Twin of `25_unit-tests_e.ipynb`. Same desk, your hands on the wheel: every
exercise compiles a suite of yours with `rustc --test` and reads the receipt —
the exact loop `cargo test` runs underneath, minus the cargo.


### Contents

- **0.** How to complete these exercises — the desk (`0.1`, given) and your first suite (`0.2`)
- **1.** The `assert` family — engineering a failure (`1.1`), the three shapes (`1.2`)
- **2.** Expected panics — `#[should_panic]` bare (`2.1`) and with `expected` (`2.2`)
- **3.** `Result`-returning tests — `?` as the assert (`3.1`)
- **4.** Choosing what runs — filters (`4.1`) and `--list` (`4.2`)
- **5.** Fixtures — setup + `Drop` teardown (`5.1`)
- **6.** Organizing — the `mod tests` idiom and the plain-build strip (`6.1`)
- **7.** The result line — the CI's parser as a function (`7.1`)
- **8.** Capstone — the five-shape matrix (`8.1`)


## 0. How to Complete These Exercises

Each exercise cell hands you a suite source — a `const ... : &str` holding a
complete `rustc --test` program — with one or more bodies stubbed to
`todo!()`, plus an immutable `// --- check ---` block that compiles and runs
it on the desk.

- **Replace every `todo!()`** so the check block's assertions hold. The doc
  comments and the section prose are the contract; Python's `unittest` mapping
  lives in §9 of the example notebook.
- **Tests must live inside the raw string.** The kernel strips `#[test]` fns
  typed at cell top level (example §7) — the desk is the only place they run.
- **Stub state is loud on purpose**: the first check aborts the run with exit
  code 101, and stub-state warnings (an import the check block does not name,
  a function only your tests call) are normal until you fill the bodies in.
- The desk writes into `scratch25/` next to this notebook; §13.1 of the
  example notebook removes it.


In [ ]:
// 0.1 Given: the desk. Every exercise compiles a string of Rust with rustc
// --test and reads the receipt -- the same loop cargo test runs underneath.
use std::fs;
use std::process::Command;

/// Write `src` to scratch25/desk_<name>.rs and compile it with rustc
/// (--test when `test_mode`). Returns the compiler's stderr: empty means
/// the build was clean.
fn desk_stderr(name: &str, src: &str, test_mode: bool) -> String {
    fs::create_dir_all("scratch25").unwrap();
    let rs = format!("scratch25/desk_{name}.rs");
    let exe = format!("scratch25/desk_{name}.exe");
    fs::write(&rs, src).unwrap();
    let mut cmd = Command::new("rustc");
    cmd.args(["--edition", "2024"]);
    if test_mode {
        cmd.arg("--test");
    }
    let out = cmd.args([&rs, "-o", &exe]).output().unwrap();
    String::from_utf8_lossy(&out.stderr).into_owned()
}

/// Compile `src` with the harness on and run the test binary with `args`.
/// Returns `(stdout, exit_code)`. Warnings pass through (stub-state suites
/// warn until their bodies are filled), but a desk source that does not
/// *compile* is a bug in this notebook and panics with the compiler's
/// output; a *test* that fails at runtime is data, like any other receipt.
fn desk_run(name: &str, src: &str, args: &[&str]) -> (String, i32) {
    let stderr = desk_stderr(name, src, true);
    assert!(
        !stderr.contains("error"),
        "desk_{name} failed to compile:\n{stderr}"
    );
    let run = Command::new(format!("./scratch25/desk_{name}.exe"))
        .args(args)
        .output()
        .unwrap();
    let code = run.status.code().unwrap_or(-1);
    (String::from_utf8_lossy(&run.stdout).into_owned(), code)
}

println!("desk helpers ready (desk_stderr, desk_run)");


### Exercise 0.2 — Your first suite

Two tests, both passing, one receipt. Fill both bodies with `assert_eq!` calls
against the given `normalize_sym`.


In [ ]:
// 0.2 Exercise: your first suite -- two passing tests, one receipt.
const NORMALIZE_SRC: &str = r#"fn normalize_sym(s: &str) -> String {
    s.trim().to_uppercase()
}

#[test]
fn trims_and_uppercases() {
    todo!()
}

#[test]
fn keeps_short_symbols() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("first", NORMALIZE_SRC, &[]);
assert!(out.contains("running 2 tests"), "out: {out}");
assert!(out.contains("test result: ok. 2 passed; 0 failed; 0 ignored"), "out: {out}");
assert_eq!(code, 0);
println!("Exercise 0.2 passed");


## 1. The `assert` Family

A failing test is not an accident to avoid — it is output you can shape. §1.1
asks you to *engineer* a failure with a custom message, then read its anatomy
out of the receipt; §1.2 cycles the three assert shapes.


In [ ]:
// 1.1 Exercise: engineer a failure, then read its anatomy. The message, the
// left value and the right value each land on their own receipt line.
const QTY_SRC: &str = r#"fn qty_of(line: &str) -> i64 {
    line.split(',').nth(1).unwrap().trim().parse().unwrap()
}

#[test]
fn parses_qty() {
    assert_eq!(qty_of("AAPL,100"), 100);
}

#[test]
fn wrong_qty() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("qtyfail", QTY_SRC, &[]);
assert!(out.contains("running 2 tests"), "out: {out}");
let start = out.find("---- wrong_qty stdout ----").unwrap();
let section = out[start..].to_string();
assert!(section.contains("assertion `left == right` failed: MSFT sells in round lots"), "section: {section}");
assert!(section.contains(" left: 50"), "section: {section}");
assert!(section.contains(" right: 60"), "section: {section}");
assert_eq!(code, 101);
println!("Exercise 1.1 passed");


### Exercise 1.2 — The three shapes

`assert_eq!` for equality, `assert_ne!` for inequality, and
`assert!(matches!(...))` for shapes — one test each, all passing.


In [ ]:
// 1.2 Exercise: one test per assert shape, all green.
const SHAPES_SRC: &str = r#"fn side_of(px: f64) -> &'static str {
    if px >= 0.0 { "BUY" } else { "SELL" }
}

#[test]
fn eq_shape() {
    todo!()
}

#[test]
fn ne_shape() {
    todo!()
}

#[test]
fn predicate_shape() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("shapes", SHAPES_SRC, &[]);
assert!(out.contains("running 3 tests"), "out: {out}");
assert!(out.contains("test result: ok. 3 passed; 0 failed; 0 ignored"), "out: {out}");
assert_eq!(code, 0);
println!("Exercise 1.2 passed");


## 2. Expected Panics: `#[should_panic]`

A `panic!` inside a test fails it — unless the test says the panic is the
point. Bare `#[should_panic]` accepts any panic; the `expected = "substring"`
form demands the message contains your text.


In [ ]:
// 2.1 Exercise: the bare form. The attribute is given; your body must
// actually panic when the line is empty.
const PANIC_SRC: &str = r#"fn first_field(line: &str) -> &str {
    assert!(!line.is_empty(), "empty line");
    line.split(',').next().unwrap()
}

#[test]
#[should_panic]
fn empty_line_panics() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("panic", PANIC_SRC, &[]);
assert!(out.contains("test empty_line_panics - should panic ... ok"), "out: {out}");
assert!(out.contains("test result: ok. 1 passed; 0 failed; 0 ignored"), "out: {out}");
assert_eq!(code, 0);
println!("Exercise 2.1 passed");


### Exercise 2.2 — The expected substring

Any panic would pass the bare form. Pin this one down: the attribute must
require the `expect` message's own text.


In [ ]:
// 2.2 Exercise: the substring, checked. Note the attribute takes a full
// `expected = "..."` item -- and the harness prints `... - should panic` on
// the receipt either way.
const PANIC_MSG_SRC: &str = r#"fn parse_qty(s: &str) -> i64 {
    s.trim().parse().expect("qty is not a number")
}

#[test]
#[should_panic(expected = "qty is not a number")]
fn rejects_text_qty() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("panicmsg", PANIC_MSG_SRC, &[]);
assert!(out.contains("test rejects_text_qty - should panic ... ok"), "out: {out}");
assert_eq!(code, 0);
assert!(!out.contains("panic did not contain expected string"), "out: {out}");
println!("Exercise 2.2 passed");


## 3. `Result`-Returning Tests

Return `Result<(), E>` and `?` becomes the assertion: an `Err` bubbles up and
the harness prints it as the failure — no panic, no `assert!`. Unwrap the Ok
side with the usual tools.


In [ ]:
// 3.1 Exercise: let `?` do the asserting.
const RESULT_SRC: &str = r#"fn parse_qty(s: &str) -> Result<i64, String> {
    s.trim().parse::<i64>().map_err(|e| e.to_string())
}

#[test]
fn reads_two_qtys() -> Result<(), String> {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("resok", RESULT_SRC, &[]);
assert!(out.contains("test reads_two_qtys ... ok"), "out: {out}");
assert!(out.contains("test result: ok. 1 passed; 0 failed; 0 ignored"), "out: {out}");
assert_eq!(code, 0);
println!("Exercise 3.1 passed");


## 4. Choosing What Runs

The example notebook's standing suite — six tests, one of each interesting
shape — is given below. It is *your* job to drive it: run it with filters,
read how many tests actually ran, and parse `--list`.


In [ ]:
// 4.0 Given: the standing suite from §5.1 of the example notebook, verbatim.
const SUITE_SRC: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    if s.is_empty() {
        return Err("empty input".to_string());
    }
    let parts: Vec<&str> = s.split(',').collect();
    if parts.len() != 2 {
        return Err(format!("expected 2 fields, got {}", parts.len()));
    }
    let sym = parts[0].trim().to_uppercase();
    let qty: i64 = parts[1]
        .trim()
        .parse()
        .map_err(|_| format!("bad qty: {}", parts[1]))?;
    Ok((sym, qty))
}

#[cfg(test)]
fn fixture_row() -> String {
    "AAPL, 100".to_string()
}

#[cfg(test)]
mod scan {
    use super::*;

    #[test]
    fn module_path_tests_work() {
        let (sym, _) = parse_fill(&fixture_row()).unwrap();
        assert_eq!(sym, "AAPL");
    }
}

#[test]
fn parses_ok() {
    let (sym, qty) = parse_fill("AAPL, 100").unwrap();
    assert_eq!(sym, "AAPL");
    assert_eq!(qty, 100);
}

#[test]
fn rejects_bad_qty() -> Result<(), String> {
    let (_, qty) = parse_fill("MSFT, 50")?;
    assert_eq!(qty, 50);
    let err = parse_fill("MSFT, lots").unwrap_err();
    assert!(err.contains("bad qty"), "got: {err}");
    Ok(())
}

#[test]
#[should_panic(expected = "empty input")]
fn panics_on_empty() {
    parse_fill("").unwrap();
}

#[test]
#[ignore = "slow stand-in for --ignored"]
fn heavy_demo() {
    std::thread::sleep(std::time::Duration::from_millis(5));
}

#[test]
fn quiet_demo() {
    println!("side effect printed from inside quiet_demo");
    parse_fill("T, 1").unwrap();
}
"#;


### Exercise 4.1 — The filter receipt

Substring filters select by full test path; a filter that matches nothing is
**not** an error — the harness runs zero tests and exits 0. Return both
numbers.


In [ ]:
// 4.1 Exercise: run the standing suite with `filter` and return
// (tests run, exit code), read off the result line.
fn filter_receipt(filter: &str) -> (usize, i32) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(filter_receipt("parse"), (1, 0), "substring: only parses_ok");
assert_eq!(filter_receipt("nope-at-all"), (0, 0), "no match still exits 0");
println!("Exercise 4.1 passed");


### Exercise 4.2 — `--list`, parsed

`--list` runs no tests; it prints one `name: test` line per test plus a
summary. Collect the names, in listing order.


In [ ]:
// 4.2 Exercise: the inventory, as data.
fn list_test_names() -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let names = list_test_names();
assert_eq!(names.len(), 6, "names: {names:?}");
assert_eq!(names[0], "heavy_demo", "names: {names:?}");
assert!(names.contains(&"scan::module_path_tests_work".to_string()), "names: {names:?}");
assert!(names.iter().all(|n| !n.contains(": test")), "suffix stripped: {names:?}");
println!("Exercise 4.2 passed");


## 5. Fixtures: Setup + Teardown

Rust has no `setUp`/`tearDown` hooks — the idiom is a value whose constructor
prints the setup and whose `Drop` prints the teardown. Each test builds its
own fixture, so state is fresh by construction, and the drop runs even when
an assert panics (unwinding calls `drop` too).


In [ ]:
// 5.1 Exercise: the fixture is given; wire up the tests. One passes, one
// fails on purpose -- the receipt must show teardown running for both.
const FIXTURE_SRC: &str = r#"struct Fixture {
    name: &'static str,
}

impl Fixture {
    fn new(name: &'static str) -> Fixture {
        println!("[setup] {name}");
        Fixture { name }
    }
}

impl Drop for Fixture {
    fn drop(&mut self) {
        println!("[teardown] {}", self.name);
    }
}

#[test]
fn fresh_state_each_test() {
    todo!()
}

#[test]
fn teardown_survives_failure() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("pfix", FIXTURE_SRC, &["--show-output"]);
assert!(out.contains("---- fresh_state_each_test stdout ----"), "out: {out}");
assert!(out.contains("[setup] fresh_state"), "out: {out}");
assert!(out.contains("[teardown] fresh_state"), "out: {out}");
let start = out.find("---- teardown_survives_failure stdout ----").unwrap();
let section = out[start..].to_string();
assert!(section.contains("[setup] on_panic"), "section: {section}");
assert!(section.contains("[teardown] on_panic"), "section: {section}");
assert!(section.contains("deliberate failure"), "section: {section}");
assert_eq!(code, 101);
println!("Exercise 5.1 passed");


## 6. Organizing: `mod tests` and the Plain-Build Strip

The cargo convention is a `#[cfg(test)] mod tests { use super::*; }` beside
the code under test. Two receipts matter: with `--test` the tests run through
their module path; without it the whole module evaporates and only the
`never used` warning remembers the lib fn existed.


In [ ]:
// 6.1 Exercise: fill both stubs, then confirm both receipts -- the test run
// and the strip.
const MOD_SRC: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    todo!()
}

fn main() {}

#[cfg(test)]
mod tests {
    use super::*;

    #[test]
    fn parses_sym_qty() {
        todo!()
    }
}
"#;

// --- check (do not modify) ---
// 1) with the harness: your test runs through the module path.
let (out, code) = desk_run("pmod", MOD_SRC, &[]);
assert!(out.contains("test tests::parses_sym_qty ... ok"), "out: {out}");
assert!(out.contains("test result: ok. 1 passed; 0 failed; 0 ignored"), "out: {out}");
assert_eq!(code, 0);
// 2) without it: the test vanishes and the lib fn is never used.
let stderr = desk_stderr("pmodprod", MOD_SRC, false);
assert!(stderr.contains("warning: function `parse_fill` is never used"), "stderr: {stderr}");
assert!(!stderr.contains("error"), "stderr: {stderr}");
println!("Exercise 6.1 passed");


## 7. The Result Line

Every run ends with `test result: ok. 5 passed; 0 failed; 1 ignored; ...` —
the line CI parsers grep for. §8.1 of the example notebook parsed it inline;
here it becomes a function the check block feeds real receipts.


In [ ]:
// 7.1 Exercise: the CI's parser, as a function. Split the counts off the
// verdict, split on ';', keep "label: number" pairs, drop the "finished"
// timing segment.
use std::collections::HashMap;

/// Parse a `test result: ...` line into its counts.
fn parse_result_line(line: &str) -> HashMap<String, u32> {
    todo!()
}

// --- check (do not modify) ---
let (out, _) = desk_run("suite", SUITE_SRC, &[]);
let line = out
    .lines()
    .find(|l| l.starts_with("test result:"))
    .unwrap()
    .to_string();
let counts = parse_result_line(&line);
assert_eq!(counts["passed"], 5, "counts: {counts:?}");
assert_eq!(counts["failed"], 0, "counts: {counts:?}");
assert_eq!(counts["ignored"], 1, "counts: {counts:?}");
assert_eq!(counts["measured"], 0, "counts: {counts:?}");
assert_eq!(counts["filtered out"], 0, "counts: {counts:?}");
assert!(counts.keys().all(|k| !k.starts_with("finished")), "counts: {counts:?}");
println!("Exercise 7.1 passed");


## 8. Capstone: the Five-Shape Matrix

One suite, five tests: a pass, a deliberate assert failure, an ignored test,
a should-panic pass and a deliberate `Err` failure. Fill the two stubs; the
given tests already fail and pause on purpose.


In [ ]:
// 8.1 Exercise: the matrix. Fill parses_ok and rejects_empty -- the other
// three shapes are given, failing and pausing on purpose.
const MATRIX_SRC: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    if s.is_empty() {
        panic!("empty input");
    }
    let parts: Vec<&str> = s.split(',').collect();
    let sym = parts[0].trim().to_uppercase();
    let qty: i64 = parts[1].trim().parse::<i64>().map_err(|e| e.to_string())?;
    Ok((sym, qty))
}

#[test]
fn parses_ok() {
    todo!()
}

#[test]
fn fails_on_assert() {
    let (_, qty) = parse_fill("MSFT, 0").unwrap();
    assert_eq!(qty, 7, "MSFT should not trade today");
}

#[test]
#[ignore = "slow stand-in for --ignored"]
fn heavy_demo() {
    std::thread::sleep(std::time::Duration::from_millis(5));
}

#[test]
#[should_panic(expected = "empty input")]
fn rejects_empty() {
    todo!()
}

#[test]
fn err_is_failure() -> Result<(), String> {
    let err = parse_fill("AAPL, lots").unwrap_err();
    Err(format!("bad qty: {err}"))
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("pmatrix", MATRIX_SRC, &["--show-output"]);
assert!(out.contains("test result: FAILED. 2 passed; 2 failed; 1 ignored"), "out: {out}");
assert!(out.contains("test parses_ok ... ok"), "out: {out}");
assert!(out.contains("test rejects_empty - should panic ... ok"), "out: {out}");
assert!(out.contains("test heavy_demo ... ignored, slow stand-in for --ignored"), "out: {out}");
assert!(out.contains("assertion `left == right` failed: MSFT should not trade today"), "out: {out}");
assert!(out.contains("Error: \"bad qty: invalid digit found in string\""), "out: {out}");
assert_eq!(code, 101);
println!("Exercise 8.1 passed");


### What Comes Next

Chapter 26 pulls the curtain: `#[test]` itself is an **attribute macro**, and
`assert_eq!` is a **declarative** one (`macro_rules!`) — the same machinery
you will write yourself.
